# 🏢 VAT Risk Analysis Workshop — Solution Notebook (เฉลย)
### การวิเคราะห์ความเสี่ยงการยกเลิก VAT ด้วย PCA + K-Means + ML — เฉลยสมบูรณ์

---

📌 **สมุดบันทึกนี้คือเฉลยสมบูรณ์ของ `vat_teaching_exercise.ipynb`**  
**This is the complete solution for `vat_teaching_exercise.ipynb`.**

---

## 🎯 โจทย์ / Business Problem
> **"วิเคราะห์ข้อมูลผู้ประกอบการ VAT เพื่อค้นหาปัจจัยเชิงพื้นที่ที่มีผลต่อความเสี่ยงการยกเลิกทะเบียน และสร้างโมเดลทำนาย"**

---
## 📦 ขั้นตอนที่ 0: นำเข้าไลบรารี / Step 0: Import Libraries

In [ ]:
# ✅ Solution — Step 0: Import Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import silhouette_score, classification_report

warnings.filterwarnings('ignore')
print("✅ Libraries imported successfully!")

---
## 🗂️ ขั้นตอนที่ 1: โหลดข้อมูล / Step 1: Load Data

In [ ]:
# ✅ Solution — Step 1: Load VAT Data
PATH_DATA = r"E:\DGA_ALL\DGA_2\Revenue_Department\For_final\data\dga306.csv"

# ลองโหลดด้วย encoding หลายแบบ / Try multiple encodings
df = None
for enc in ['tis-620', 'utf-8-sig', 'utf-8', 'cp874']:
    try:
        df = pd.read_csv(PATH_DATA, encoding=enc, encoding_errors='ignore', low_memory=False)
        df.columns = df.columns.str.strip()
        print(f"✅ Loaded with encoding: {enc}")
        break
    except Exception as e:
        print(f"   Failed with {enc}: {e}")

if df is not None:
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("Columns:", df.columns.tolist())
else:
    print("❌ ไม่สามารถโหลดข้อมูลได้ / Could not load data")

---
## 🔭 ขั้นตอนที่ 2: EDA / Step 2: Exploratory Data Analysis

In [ ]:
# ✅ Solution — Step 2.1: Preview data
display(df.head())

In [ ]:
# ✅ Solution — Step 2.2: Data structure
df.info()

In [ ]:
# ✅ Solution — Step 2.3: Missing values
missing = pd.DataFrame({
    'Missing_Count': df.isna().sum(),
    'Missing_%': (df.isna().sum() / len(df) * 100).round(2)
})
display(missing)

---
## 🧹 ขั้นตอนที่ 3: ทำความสะอาดข้อมูล / Step 3: Data Cleaning

In [ ]:
# ✅ Solution — Step 3.1: Convert BE date to CE date
def clean_be_date(date_str):
    """แปลงวันที่ พ.ศ. → ค.ศ. / Convert Buddhist Era date to CE Timestamp."""
    if pd.isna(date_str) or not isinstance(date_str, str):
        return pd.NaT
    try:
        parts = date_str.strip().split('-')
        if len(parts) == 3:
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            if year > 2400:  # พ.ศ. → ค.ศ. (ลบ 543)
                year -= 543
            return pd.Timestamp(year=year, month=month, day=day)
    except Exception:
        pass
    return pd.NaT

# ตรวจสอบว่ามีคอลัมน์วันที่ / Find date column
date_col = 'วันที่ได้รับอนุมัติ'
if date_col not in df.columns:
    # ลองหาคอลัมน์ที่ใกล้เคียง
    date_col = [c for c in df.columns if 'วัน' in c or 'date' in c.lower()]
    date_col = date_col[0] if date_col else None
    print(f"Using column: {date_col}")

if date_col:
    df['registration_date'] = df[date_col].apply(clean_be_date)
    df = df.dropna(subset=['registration_date'])
    print(f"✅ Rows after date cleaning: {len(df):,}")
else:
    print("❌ Date column not found — creating dummy date")
    df['registration_date'] = pd.Timestamp('2010-01-01')

In [ ]:
# ✅ Solution — Step 3.2: Calculate business age + clean whitespace
ref_date = pd.Timestamp('2026-07-01')
df['business_age_years'] = (ref_date - df['registration_date']).dt.days / 365.25
df = df[df['business_age_years'] >= 0]  # กรองค่าผิดปกติ

# ทำความสะอาดช่องว่างส่วนเกิน
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.strip()

print(f"อายุธุรกิจเฉลี่ย / Avg business age: {df['business_age_years'].mean():.1f} years")
print(f"สูงสุด / Max: {df['business_age_years'].max():.1f} | ต่ำสุด / Min: {df['business_age_years'].min():.1f}")

plt.figure(figsize=(10, 4))
plt.hist(df['business_age_years'].clip(0, 40), bins=40, color='steelblue', edgecolor='white')
plt.title('Distribution of Business Age / การกระจายอายุธุรกิจ')
plt.xlabel('Business Age (years)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## ⚙️ ขั้นตอนที่ 4: Feature Engineering / Step 4: Feature Engineering

In [ ]:
# ✅ Solution — Step 4.1: Business density by postcode
# หาคอลัมน์รหัสไปรษณีย์
postcode_col = 'รหัสไปรษณีย์'
if postcode_col not in df.columns:
    postcode_col = [c for c in df.columns if 'ไปรษณีย์' in c or 'zip' in c.lower() or 'post' in c.lower()]
    postcode_col = postcode_col[0] if postcode_col else None

if postcode_col:
    postcode_counts = df[postcode_col].value_counts().to_dict()
    df['postcode_business_density'] = df[postcode_col].map(postcode_counts)
    print("Postcode density stats:")
    print(df['postcode_business_density'].describe())
else:
    print("⚠️ Postcode column not found — using district density instead")
    # fallback: use อำเภอ density
    if 'อำเภอ' in df.columns:
        district_counts = df['อำเภอ'].value_counts().to_dict()
        df['postcode_business_density'] = df['อำเภอ'].map(district_counts)
    else:
        df['postcode_business_density'] = 1

In [ ]:
# ✅ Solution — Step 4.2: Sample 100,000 rows
df_sample = df.sample(n=min(100_000, len(df)), random_state=42).reset_index(drop=True)
print(f"✅ Sample size: {len(df_sample):,} rows")
display(df_sample[['business_age_years', 'postcode_business_density']].head())

---
## 📉 ขั้นตอนที่ 5: PCA / Step 5: PCA

In [ ]:
# ✅ Solution — Step 5.1: Scale features
feature_cols = ['business_age_years', 'postcode_business_density']
X_raw = df_sample[feature_cols].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"✅ Data scaled. Shape: {X_scaled.shape}")
print(f"Means after scaling (should be ~0): {X_scaled.mean(axis=0).round(5)}")
print(f"STDs after scaling (should be ~1):  {X_scaled.std(axis=0).round(5)}")

In [ ]:
# ✅ Solution — Step 5.2: PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"=== PCA Results ===")
print(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Explained Variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.15, s=3, color='steelblue')
plt.title(
    f'PCA 2D Projection of VAT Business Data\n'
    f'(Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%)'
)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout()
plt.show()

---
## 🎯 ขั้นตอนที่ 6: K-Means Clustering / Step 6: K-Means

In [ ]:
# ✅ Solution — Step 6.1: Elbow + Silhouette
wcss = []
sil  = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(k_range, wcss, marker='o', linewidth=2)
axes[0].set_title('Elbow Method / วิธี Elbow')
axes[0].set_xlabel('k')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].set_xticks(k_range)
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(k_range, sil, marker='s', linewidth=2, color='green')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Score')
axes[1].set_xticks(k_range)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Finding Optimal Number of Clusters (k)\nหาจำนวนกลุ่มที่เหมาะสม', y=1.02)
plt.tight_layout()
plt.show()

best_k = list(k_range)[sil.index(max(sil))]
print(f"\n✅ Best k by Silhouette Score: k={best_k} (score={max(sil):.4f})")

In [ ]:
# ✅ Solution — Step 6.2: Final clustering + PCA plot
BEST_K = best_k

km_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled)

df_sample_cluster = df_sample.loc[X_raw.index].copy()
df_sample_cluster['Cluster'] = cluster_labels

colors = plt.cm.tab10.colors
plt.figure(figsize=(9, 6))
for c in range(BEST_K):
    mask = cluster_labels == c
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        label=f'Cluster {c}', alpha=0.4, s=6, color=colors[c]
    )
plt.title(f'K-Means (k={BEST_K}) on PCA Space\nกลุ่มบน PCA 2D')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.show()

print("\n📊 Cluster Characteristics (mean values):")
display(df_sample_cluster.groupby('Cluster')[['business_age_years', 'postcode_business_density']].mean().round(1))

---
## 🤖 ขั้นตอนที่ 7: ML — Random Forest Classifier / Step 7: ML

In [ ]:
# ✅ Solution — Step 7.1: Create target variable
# สมมติ: ธุรกิจอายุ < 3 ปี = ความเสี่ยงสูง / Assume age < 3 years = high risk
df_sample_cluster['high_risk'] = (df_sample_cluster['business_age_years'] < 3).astype(int)

print("Target distribution / การกระจายของ Target:")
vc = df_sample_cluster['high_risk'].value_counts()
print(vc)
print(f"\nHigh Risk %: {vc.get(1, 0) / vc.sum() * 100:.1f}%")
print(f"Low Risk  %: {vc.get(0, 0) / vc.sum() * 100:.1f}%")

In [ ]:
# ✅ Solution — Step 7.2: Train Random Forest Classifier
X_ml = df_sample_cluster[['business_age_years', 'postcode_business_density']].dropna()
y_ml = df_sample_cluster.loc[X_ml.index, 'high_risk']

# แบ่ง train/test 80/20 + stratify เพื่อรักษาสัดส่วน target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=42, stratify=y_ml
)

print(f"Train size: {len(X_tr):,} | Test size: {len(X_te):,}")

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print("\n=== Classification Report ===")
print(classification_report(y_te, y_pred, target_names=['Low Risk (0)', 'High Risk (1)']))

In [ ]:
# ✅ Solution — Step 7.3: Feature Importance
importances = pd.Series(clf.feature_importances_, index=X_ml.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Feature importance bar
importances.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Feature Importance — Random Forest')
axes[0].set_xlabel('Importance')

# Cluster vs risk
risk_by_cluster = df_sample_cluster.groupby('Cluster')['high_risk'].mean() * 100
risk_by_cluster.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('High Risk % by Cluster\n% ความเสี่ยงสูงในแต่ละกลุ่ม')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('% High Risk')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

print("\nHigh Risk % per cluster:")
print(risk_by_cluster.round(1).to_frame())

---
## 🏁 สรุปผลการวิเคราะห์ / Analysis Summary

| หัวข้อ | ผลที่ได้ |
|---|---|
| **ขนาดข้อมูล** | ~1.1 ล้านแถว, ข้อมูล VAT ผู้ประกอบการ |
| **Data Cleaning** | แปลงวันที่ พ.ศ. → ค.ศ., คำนวณอายุธุรกิจ |
| **Feature Engineering** | ความหนาแน่นธุรกิจตามรหัสไปรษณีย์ |
| **PCA** | ลดมิติเพื่อ visualize ใน 2D |
| **K-Means** | จัดกลุ่มธุรกิจตามอายุและความหนาแน่น |
| **Random Forest** | ทำนายความเสี่ยงการยกเลิก VAT |

📂 **ต้องการฝึกหัด?** เปิดไฟล์ `vat_teaching_exercise.ipynb`  
📂 **Want to practice?** Open `vat_teaching_exercise.ipynb`